# KawanIsyarat — Model Training Pipeline

In [1]:
# ==========================================
# CELL 1: SETUP & DEPENDENCIES
# ==========================================
!pip install mediapipe ultralytics tensorflow --quiet

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Paksa CPU agar export TFLite mulus

import cv2
import json
import numpy as np
import mediapipe as mp
from pathlib import Path
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TF version:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


2026-04-08 17:32:22.829736: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775669543.058804      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775669543.121642      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775669543.663611      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775669543.663653      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775669543.663656      23 computation_placer.cc:177] computation placer alr

TF version: 2.19.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


2026-04-08 17:32:45.881146: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


## Section 2: Model 1 — WL-BISINDO word gesture (MediaPipe + LSTM)

In [2]:
# ==========================================
# CELL 2: MEDIAPIPE + SIFORMER NORMALIZATION
# ==========================================
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
import urllib.request

# Download model MediaPipe
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
    "hand_landmarker.task"
)
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task",
    "pose_landmarker.task"
)

# Setup Landmarker
hand_options = vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="hand_landmarker.task"),
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
hand_landmarker = vision.HandLandmarker.create_from_options(hand_options)

pose_options = vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="pose_landmarker.task"),
    min_pose_detection_confidence=0.5
)
pose_landmarker = vision.PoseLandmarker.create_from_options(pose_options)

def normalize_siformer_style(pose_result, hand_result):
    """
    Ekstrak dan normalisasi ala Boháček (Siformer).
    Hanya mengambil 12 titik body + 21 titik L_hand + 21 titik R_hand.
    Output: Array (108,)  -> Jauh lebih ringan dari 258!
    """
    body_features = np.zeros(24) # 12 keypoints * 2 (X, Y)
    l_hand_features = np.zeros(42) # 21 keypoints * 2
    r_hand_features = np.zeros(42)

    # 1. BODY NORMALIZATION
    if pose_result.pose_landmarks:
        lms = pose_result.pose_landmarks[0]
        # MediaPipe Pose mapping yang kita butuhkan:
        # 0:nose, 2:L_eye, 5:R_eye, 7:L_ear, 8:R_ear
        # 11:L_shoulder, 12:R_shoulder, 13:L_elbow, 14:R_elbow, 15:L_wrist, 16:R_wrist
        
        neck_x = (lms[11].x + lms[12].x) / 2
        neck_y = (lms[11].y + lms[12].y) / 2
        
        # Hitung head_metric (Jarak antar bahu)
        head_metric = np.sqrt((lms[11].x - lms[12].x)**2 + (lms[11].y - lms[12].y)**2)
        if head_metric == 0: # Fallback jika bahu tidak terdeteksi sempurna
            head_metric = np.sqrt((neck_x - lms[0].x)**2 + (neck_y - lms[0].y)**2)

        if head_metric > 0:
            # Bounding box tubuh atas Siformer
            start_x = neck_x - 3 * head_metric
            start_y = lms[2].y + head_metric      # Patokan mata kiri
            end_x = neck_x + 3 * head_metric
            end_y = start_y - 6 * head_metric

            body_pts = [
                (lms[0].x, lms[0].y), (neck_x, neck_y), (lms[5].x, lms[5].y),
                (lms[2].x, lms[2].y), (lms[8].x, lms[8].y), (lms[7].x, lms[7].y),
                (lms[12].x, lms[12].y), (lms[11].x, lms[11].y), (lms[14].x, lms[14].y),
                (lms[13].x, lms[13].y), (lms[16].x, lms[16].y), (lms[15].x, lms[15].y)
            ]

            norm_body = []
            for x, y in body_pts:
                # Normalisasi min-max lokal
                nx = (x - start_x) / (end_x - start_x) if end_x != start_x else 0
                ny = (y - end_y) / (start_y - end_y) if start_y != end_y else 0
                norm_body.extend([nx, ny])
            body_features = np.array(norm_body)

    # 2. HAND NORMALIZATION (Fungsi lokal)
    def normalize_hand(hand_lms):
        xs = [lm.x for lm in hand_lms]
        ys = [lm.y for lm in hand_lms]
        width, height = max(xs) - min(xs), max(ys) - min(ys)
        
        # Buat box menjadi persegi proporsional
        if width > height:
            delta_x = 0.1 * width
            delta_y = delta_x + ((width - height) / 2)
        else:
            delta_y = 0.1 * height
            delta_x = delta_y + ((height - width) / 2)

        start_x, end_x = min(xs) - delta_x, max(xs) + delta_x
        start_y, end_y = min(ys) - delta_y, max(ys) + delta_y

        norm_hand = []
        for lm in hand_lms:
            nx = (lm.x - start_x) / (end_x - start_x) if end_x != start_x else 0
            ny = (lm.y - start_y) / (end_y - start_y) if end_y != start_y else 0
            norm_hand.extend([nx, ny])
        return np.array(norm_hand)

    if hand_result.hand_landmarks:
        for i, hand_lms in enumerate(hand_result.hand_landmarks):
            if i < len(hand_result.handedness):
                label = hand_result.handedness[i][0].category_name
                if label == "Left":
                    l_hand_features = normalize_hand(hand_lms)
                else:
                    r_hand_features = normalize_hand(hand_lms)

    return np.concatenate([body_features, l_hand_features, r_hand_features]) # Length: 108

def video_to_keypoints(video_path, sequence_length=30):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        cap.release()
        return np.zeros((sequence_length, 108))
    
    indices = np.linspace(0, total_frames - 1, sequence_length, dtype=int)
    keypoints_seq = []
    
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            keypoints_seq.append(np.zeros(108))
            continue
        
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        
        hand_res = hand_landmarker.detect(mp_image)
        pose_res = pose_landmarker.detect(mp_image)
        
        features = normalize_siformer_style(pose_res, hand_res)
        keypoints_seq.append(features)
    
    cap.release()
    while len(keypoints_seq) < sequence_length:
        keypoints_seq.append(np.zeros(108))
        
    return np.array(keypoints_seq[:sequence_length])

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1775669566.265274      83 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775669566.289425      83 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775669566.387117      88 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775669566.423617      88 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [3]:
# ==========================================
# CELL 3: PROCESS DATASET & CACHE
# ==========================================
LABEL_MAP = {
    0: "Air", 1: "Belajar", 2: "Cari", 3: "Hari", 4: "Ingat", 5: "Lagi", 
    6: "Maaf", 7: "Makan", 8: "Motor", 9: "Saya", 10: "Terima kasih", 11: "Tuli",
    12: "Apa", 13: "Siapa", 14: "Kapan", 15: "Di mana", 16: "Mengapa", 17: "Bagaimana", 
    18: "Merah", 19: "Kuning", 20: "Hijau", 21: "Hitam", 22: "Dengar", 23: "Berangkat",
    24: "Datang", 25: "Teman", 26: "Keluarga", 27: "Rumah", 28: "Pagi", 29: "Siang", 
    30: "Sore", 31: "Malam"
}

# Sesuaikan DATASET_PATH dengan direktori Kaggle-mu
DATASET_PATH = "/kaggle/input/datasets/glennleonali/wl-bisindo"  
CACHE_PATH   = "/kaggle/working/keypoints_cache"
os.makedirs(CACHE_PATH, exist_ok=True)

def process_wl_bisindo_fixed(dataset_path, cache_path, sequence_length=30):
    cache_file = os.path.join(cache_path, "wl_bisindo_siformer_cache.npz")
    
    if os.path.exists(cache_file):
        print("Loading dari cache Siformer...")
        data = np.load(cache_file, allow_pickle=True)
        return data['X'], data['y'], data['label_map'].item()
    
    X, y = [], []
    video_files = list(Path(dataset_path).rglob("*.mp4"))
    print(f"Mengekstrak {len(video_files)} video. Fitur baru = 108 dimensi (Super Ringan!)...")
    
    for i, video_path in enumerate(video_files):
        parts = video_path.stem.split("_")
        try:
            label_id = int(parts[1].replace("label", ""))
            keypoints = video_to_keypoints(str(video_path), sequence_length)
            X.append(keypoints)
            y.append(label_id)
            if (i + 1) % 100 == 0:
                print(f"  {i+1}/{len(video_files)} done...")
        except Exception as e:
            continue
            
    X, y = np.array(X), np.array(y)
    np.savez_compressed(cache_file, X=X, y=y, label_map=LABEL_MAP)
    print(f"Selesai! Shape X: {X.shape}")
    return X, y, LABEL_MAP

X, y, label_map = process_wl_bisindo_fixed(DATASET_PATH, CACHE_PATH)

Mengekstrak 1600 video. Fitur baru = 108 dimensi (Super Ringan!)...


W0000 00:00:1775669572.197943      83 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


  100/1600 done...
  200/1600 done...
  300/1600 done...
  400/1600 done...
  500/1600 done...
  600/1600 done...
  700/1600 done...
  800/1600 done...
  900/1600 done...
  1000/1600 done...
  1100/1600 done...
  1200/1600 done...
  1300/1600 done...
  1400/1600 done...
  1500/1600 done...
  1600/1600 done...
Selesai! Shape X: (1600, 30, 108)


In [4]:
# ==========================================
# CELL 4: SPLITTING
# ==========================================
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.111, stratify=y_temp, random_state=42
)

print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")
print(f"Test  : {X_test.shape}")

Train : (1280, 30, 108)
Val   : (160, 30, 108)
Test  : (160, 30, 108)


In [5]:
# ==========================================
# CELL 5: MODEL & TRAINING
# ==========================================
NUM_CLASSES     = len(LABEL_MAP)
SEQUENCE_LENGTH = 30
FEATURE_DIM     = 108  # <-- Jauh lebih kecil karena kita pakai Siformer method

def build_gesture_model(num_classes, seq_len, feat_dim):
    model = Sequential([
        Input(shape=(seq_len, feat_dim)),
        
        LSTM(128, return_sequences=True, unroll=True),
        BatchNormalization(),
        Dropout(0.3),
        
        LSTM(64, return_sequences=False, unroll=True),
        BatchNormalization(),
        Dropout(0.3),
        
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_gesture_model(NUM_CLASSES, SEQUENCE_LENGTH, FEATURE_DIM)
model.summary()

callbacks = [
    EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=7, factor=0.5, min_lr=1e-6, verbose=1),
    ModelCheckpoint("/kaggle/working/best_gesture_model_siformer.keras", save_best_only=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 128)        │       121,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 30, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         4,128 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 184,480 (720.62 KB)

 Trainable params: 183,840 (718.12 KB)

 Non-trainable params: 640 (2.50 KB)

Epoch 1/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 70ms/step - accuracy: 0.1045 - loss: 3.5642 - val_accuracy: 0.1813 - val_loss: 3.3032 - learning_rate: 0.0010
Epoch 2/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.3148 - loss: 2.4272 - val_accuracy: 0.3562 - val_loss: 3.0217 - learning_rate: 0.0010
Epoch 3/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.5271 - loss: 1.6448 - val_accuracy: 0.3500 - val_loss: 2.7179 - learning_rate: 0.0010
Epoch 4/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.6228 - loss: 1.2695 - val_accuracy: 0.4000 - val_loss: 2.3315 - learning_rate: 0.0010
Epoch 5/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.7163 - loss: 0.9561 - val_accuracy: 0.3375 - val_loss: 2.2601 - learning_rate: 0.0010
Epoch 6/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.7336 - loss: 0.8425 - val_accuracy: 0.4875 - val_loss: 1.7044 - learning_rate: 0.0010
Epoch 7/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.7847 - loss: 0.7170 - 

In [6]:
# ==========================================
# CELL 6: EVALUATE & EXPORT
# ==========================================
from sklearn.metrics import classification_report

y_pred = np.argmax(model.predict(X_test), axis=1)
print("\n=== Test Set Evaluation ===")
print(classification_report(
    y_test, y_pred,
    target_names=[label_map[i] for i in sorted(label_map.keys())]
))

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = "/kaggle/working/bisindo_gesture_siformer.tflite"
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(tflite_path) / 1024 / 1024
print(f"\nTFLite size: {size_mb:.2f} MB (Sangat aman untuk Android!)")

with open("/kaggle/working/bisindo_labels.json", "w") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)
print("Labels saved! Siap dideploy ke Flutter/Android via MLKit / TFLite Interpreter.")

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step

=== Test Set Evaluation ===
              precision    recall  f1-score   support

         Air       1.00      0.80      0.89         5
     Belajar       1.00      1.00      1.00         5
        Cari       1.00      1.00      1.00         5
        Hari       1.00      1.00      1.00         5
       Ingat       1.00      1.00      1.00         5
        Lagi       1.00      1.00      1.00         5
        Maaf       1.00      1.00      1.00         5
       Makan       1.00      1.00      1.00         5
       Motor       1.00      1.00      1.00         5
        Saya       1.00      1.00      1.00         5
Terima kasih       1.00      1.00      1.00         5
        Tuli       1.00      1.00      1.00         5
         Apa       1.00      1.00      1.00         5
       Siapa       1.00      1.00      1.00         5
       Kapan       1.00      1.00      1.00         5
     Di mana       1.00      1.00      1.00         5
     Mengapa  

INFO:tensorflow:Assets written to: /tmp/tmp3bg6s8eq/assets


Saved artifact at '/tmp/tmp3bg6s8eq'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 30, 108), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 32), dtype=tf.float32, name=None)
Captures:
  133903150138576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133903150141840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133903150143376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133903150143760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133903150139152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133903150142992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133903150143184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133903150144336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133906859548816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133903150142032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133903150144144: 

W0000 00:00:1775685555.911292      23 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1775685555.911354      23 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1775685555.946908      23 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled



TFLite size: 0.86 MB (Sangat aman untuk Android!)
Labels saved! Siap dideploy ke Flutter/Android via MLKit / TFLite Interpreter.
